## Lesson Overview

**What this lesson teaches:** the *minimal* version of a tool-calling loop — a single tool, so you can see the round-trip mechanics without the registry/dispatch complexity that Lesson 7 adds on top.

**What's happening under the hood, step by step:**
1. **Setup cell** loads `.env` and creates the shared `client`/`MODEL_NAME`, identical pattern to every other lesson notebook.
2. **One tool is defined**, `get_current_datetime`, alongside its JSON `input_schema` — the description and parameter list the model reads to decide *if* and *how* to call it.
3. **`run_tool`** dispatches by name to the matching Python function. With only one tool this is just an `if`, but it's the same idea as Lesson 7's `TOOL_REGISTRY`/`dispatch_tool`, just not generalized yet.
4. **`run_tools`** scans the model's response for `tool_use` blocks, calls `run_tool` for each, and wraps the outcome as a `tool_result` block — success or a caught exception turned into `is_error: True` — ready to send back.
5. **`run_conversation`** is the loop itself: call `chat()` → print whatever text came back → if `response.stop_reason` isn't `"tool_use"`, the model is done, so stop → otherwise get results from `run_tools`, append them as the next user turn, and loop again.

The pattern to internalize: Claude never executes anything itself — it only *asks* for a tool call. Your Python code is what actually runs the function and hands the result back, and the "conversation" is really just this request → run → report cycle repeating until the model has what it needs to give a final answer. Lesson 7 builds on exactly this loop, just with more tools and a shared dispatcher.


# Lesson 8: Tool use with a current-time helper

This notebook demonstrates a simple tool-calling loop with Anthropic:
- it loads your local `.env` file
- it creates a client from `ANTHROPIC_API_KEY` and `MODEL_NAME`
- it defines one callable tool for reading the current time
- it runs a short conversation that can trigger multiple tool calls

Use it as a guided training notebook: read each explanation block before the code cell that follows it, then compare the printed output with the schema and helper definitions.

The main things to watch are the message format, the tool schema, the assistant's tool request, and the round trip back into Python.

The notebook is written so the setup and local helper cells still make sense on their own, while the final API demo only runs when the Anthropic package and key are available.


## How This Notebook Works

The lesson follows a simple pattern:
1. load environment variables and build the Anthropic client
2. define helper functions that keep message handling consistent
3. define the tool and its JSON schema
4. run a conversation loop that lets the model decide whether to call the tool
5. inspect the local output so you can see what Python returned

This layout is intentional. It lets you understand the moving parts separately before you combine them into a full tool-calling flow.


## Setup

Your `.env` file is already in place. The notebook looks for it in the project root and in `Claude_API_Training/.env`.

The expected variables are:

```env
ANTHROPIC_API_KEY=...
MODEL_NAME=claude-haiku-4-5
```

If `MODEL_NAME` is missing, the notebook falls back to `claude-haiku-4-5`.


In [1]:
import json
import os
from datetime import datetime, timedelta
from pathlib import Path

try:
    import anthropic
except ImportError:
    anthropic = None

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

def load_env_file(env_path):
    env_path = Path(env_path)
    if not env_path.exists():
        return False

    with env_path.open('r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            key, value = line.split('=', 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")
    return True

load_dotenv()

env_loaded = False
for candidate in (Path('.env'), Path('Claude_API_Training/.env')):
    if load_env_file(candidate):
        env_loaded = True
        break

API_KEY = os.environ.get('ANTHROPIC_API_KEY')
MODEL_NAME = os.environ.get('MODEL_NAME', 'claude-haiku-4-5')
model = MODEL_NAME
client = anthropic.Anthropic(api_key=API_KEY) if anthropic and API_KEY else None

print(f'Environment loaded: {env_loaded}')
print(f'Model: {MODEL_NAME}')
if client is None:
    print('Anthropic client is not available in this runtime yet. The local helper cells will still run.')
else:
    print('Anthropic client ready.')


Environment loaded: True
Model: claude-haiku-4-5-20251001
Anthropic client ready.


### Setup Notes

The setup cell is the only place where environment state is loaded. After it runs, every other cell can reuse the same client and model name.

If the package is not installed in the active environment, the notebook still keeps the local helper cells readable, but the API conversation cell will skip cleanly.


## Message helpers

Anthropic conversations are tracked as `role` and `content` dictionaries. The helper functions below keep that shape consistent across user turns, assistant turns, and tool results.


In [2]:
def add_user_message(messages, message):
    user_message = {
        'role': 'user',
        'content': message.content if hasattr(message, 'content') else message,
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
        'role': 'assistant',
        'content': message.content if hasattr(message, 'content') else message,
    }
    messages.append(assistant_message)

def text_from_message(message):
    return "\n".join(
        block.text for block in message.content if getattr(block, 'type', None) == 'text'
    )

def chat(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):
    if client is None:
        raise RuntimeError('Set up the Anthropic package and ANTHROPIC_API_KEY before calling chat().')

    params = {
        'model': model,
        'max_tokens': 1000,
        'messages': messages,
        'temperature': temperature,
        'stop_sequences': stop_sequences or [],
    }

    if tools:
        params['tools'] = tools
    if system:
        params['system'] = system

    return client.messages.create(**params)


### Why These Helpers Matter

The helper functions are small on purpose. They reduce the chance of formatting errors when you build up a conversation over multiple turns.

`add_user_message` and `add_assistant_message` keep the history list consistent. `text_from_message` turns the model response into something easy to read in the notebook output.


## Tool and schema

This lesson keeps the tool surface intentionally small. The model can ask for the current time in a requested format, and the notebook returns the result through the tool loop.


In [3]:
def get_current_datetime(date_format='%Y-%m-%d %H:%M:%S'):
    if not date_format:
        raise ValueError('date_format cannot be empty')
    return datetime.now().strftime(date_format)

get_current_datetime_schema = {
    'name': 'get_current_datetime',
    'description': 'Returns the current date and time formatted according to the requested format string.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'date_format': {
                'type': 'string',
                'description': 'Python strftime format string used for the output.',
                'default': '%Y-%m-%d %H:%M:%S',
            }
        },
        'required': [],
    },
}

def run_tool(tool_name, tool_input):
    if tool_name == 'get_current_datetime':
        return get_current_datetime(**tool_input)
    raise ValueError(f'Unknown tool: {tool_name}')

def run_tools(message):
    tool_requests = [block for block in message.content if getattr(block, 'type', None) == 'tool_use']
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                'type': 'tool_result',
                'tool_use_id': tool_request.id,
                'content': json.dumps(tool_output),
                'is_error': False,
            }
        except Exception as exc:
            tool_result_block = {
                'type': 'tool_result',
                'tool_use_id': tool_request.id,
                'content': f'Error: {exc}',
                'is_error': True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks


### Tool Schema Walkthrough

The schema tells the model what the tool does and which arguments it accepts. In this lesson, `date_format` is the only input.

That keeps the example focused. The assistant can request a specific time format, Python runs the function locally, and the tool result goes back to the model.


## Conversation loop

This loop keeps calling the model until it stops asking for tools. It is the piece that makes the notebook feel usable instead of being a single one-off API request.


In [4]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != 'tool_use':
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages


### Conversation Loop

The loop is the core of the lesson. Each pass does three things:
- send the current message history to the model
- print the assistant response
- if the model asked for a tool, run it locally and send the result back

That back-and-forth is the part to understand before extending the notebook to more tools.


In [5]:
if client is None:
    print('Skipping the API demo. Install anthropic and keep ANTHROPIC_API_KEY in .env to run this cell.')
else:
    messages = []
    add_user_message(
        messages,
        'What is the current time in HH:MM format? Also, what is the current time in SS format?',
    )

    run_conversation(messages)



The current time is:
- **HH:MM format**: 18:45
- **SS format**: 23 (seconds)


## Local sanity check

These checks only use the local helper functions and schema objects. They do not need the Anthropic client.


In [6]:
print('Current time example:', get_current_datetime('%H:%M'))
print('Seconds example:', get_current_datetime('%S'))
print('Tool schema name:', get_current_datetime_schema['name'])
print('Tool schema fields:', list(get_current_datetime_schema['input_schema']['properties'].keys()))


Current time example: 18:45
Seconds example: 25
Tool schema name: get_current_datetime
Tool schema fields: ['date_format']


### What To Verify Here

This is the last quick check before you treat the notebook as a working training example.

You should see the current time formatted in two ways, plus the schema name and fields. That confirms the helper function and the tool definition match the lesson.


## Summary: how to use this notebook

1. Run the setup cell first so the notebook loads `.env` and creates the Anthropic client.
2. Run the message helper and tool cells to confirm the notebook state is ready for a conversation loop.
3. Run the final prompt cell to let the model call `get_current_datetime` in whatever format it needs.
4. If you want to change the lesson, start by extending the tool schema and then update `run_tool` to dispatch the new tool name.

The main feature here is the round trip between assistant tool requests, local Python execution, and the assistant’s final response.
